In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip -q install -U --no-cache-dir transformers accelerate bitsandbytes
!pip -q install --no-cache-dir --force-reinstall "Pillow==10.2.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 92.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 266.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 234.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 360.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.1.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 55.6 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes

In [2]:
!pip -q install -U open_clip_torch

# print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
✅ Dependencies installed


In [3]:
# ============================================================================
# CELL 2: Import all libraries
# ============================================================================
import os
import re
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import open_clip
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

print("✅ All libraries imported")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ All libraries imported


In [9]:
ROOT = "/kaggle/input/shared-poli-dataset/dataset"
TAMIL_IMG_DIR = os.path.join(ROOT, "Train_images(Tamil)")
MAL_IMG_DIR   = os.path.join(ROOT, "Train_images(Malayam)")  # note: 'Malayam' in folder name

TAMIL_XLSX = os.path.join(ROOT, "Tamil_Train_labels.xlsx")
MAL_XLSX   = os.path.join(ROOT, "Malayalam_Train_label.xlsx")

In [10]:
# -------------------------
# 1) Load labels
# -------------------------
t = pd.read_excel(TAMIL_XLSX)
m = pd.read_excel(MAL_XLSX)

# Standardize columns
t = t.rename(columns={"Level1":"Level 1", "Level2":"Level 2"})
t["lang"] = "tamil"
m["lang"] = "malayalam"

# Tamil has Image_name; Malayalam has meme_id
# We'll create an image file name for both.
def tamil_candidates(row):
    # Most robust: try Image_name then also try numeric id as jpg
    cands = []
    if "Image_name" in row and pd.notna(row["Image_name"]):
        cands.append(str(row["Image_name"]).strip())
    if "Image_id" in row and pd.notna(row["Image_id"]):
        cands.append(f"{int(row['Image_id'])}.jpg")
    # Also try 3-digit zero-pad from Image_id
    if "Image_id" in row and pd.notna(row["Image_id"]):
        cands.append(f"{int(row['Image_id']):03d}.jpg")
    return list(dict.fromkeys(cands))

def mal_candidates(row):
    # Many images in folder appear as 208.jpg etc.
    # meme_id seems numeric -> "{id}.jpg"
    cands = []
    if "meme_id" in row and pd.notna(row["meme_id"]):
        cands.append(f"{int(row['meme_id'])}.jpg")
        # some datasets may also use zero-pad, so try just in case
        cands.append(f"{int(row['meme_id']):03d}.jpg")
    return list(dict.fromkeys(cands))

def resolve_path(img_dir, candidates):
    for fn in candidates:
        p = os.path.join(img_dir, fn)
        if os.path.exists(p):
            return p
    return None

# Build resolved image paths
t["image_path"] = t.apply(lambda r: resolve_path(TAMIL_IMG_DIR, tamil_candidates(r)), axis=1)
m["image_path"] = m.apply(lambda r: resolve_path(MAL_IMG_DIR,   mal_candidates(r)), axis=1)

In [11]:
# -------------------------
# 2) Map to INTERNAL labels
# -------------------------
def map_internal(lang, level1, level2):
    # Level1 internal
    if lang == "tamil":
        stance = "SUPPORT" if level1 == "Support/Praise" else "TROLL"
        # Level2 internal
        if level2 == "Support for person" or level2 == "Troll/Oppose Against Person":
            target = "PERSON"
        elif level2 == "Support for party" or level2 == "Troll/Oppose Against Party":
            target = "PARTY"
        else:
            target = "UNKNOWN"
    else:
        stance = "SUPPORT" if str(level1).strip().upper() == "SUPPORT" else "TROLL"
        l2 = str(level2).strip()
        if l2 in ["Against individual person", "Support for individual person"]:
            target = "PERSON"
        elif l2 in ["Against party", "Support for party"]:
            target = "PARTY"
        elif l2 == "Intersection":
            target = "INTERSECTION"
        else:
            target = "UNKNOWN"
    return stance, target

def add_internal(df):
    stances, targets = [], []
    for _, r in df.iterrows():
        stance, target = map_internal(r["lang"], r["Level 1"], r["Level 2"])
        stances.append(stance); targets.append(target)
    df["stance_internal"] = stances
    df["target_internal"] = targets
    return df

t = add_internal(t)
m = add_internal(m)

# Keep only key columns
t_out = t[["lang", "image_path", "Level 1", "Level 2", "stance_internal", "target_internal"]].copy()
m_out = m[["lang", "image_path", "Level 1", "Level 2", "stance_internal", "target_internal"]].copy()

df = pd.concat([t_out, m_out], ignore_index=True)

# Quick sanity checks
missing = df["image_path"].isna().sum()
print("Total rows:", len(df))
print("Missing image_path:", missing)
if missing:
    print(df[df["image_path"].isna()].head(10))

Total rows: 1303
Missing image_path: 0


In [12]:
# -------------------------
# 3) Face detection (fast baseline)
# -------------------------
# Using OpenCV Haar cascade (fast and lightweight).
# Not best, but good enough for "face count" signal on Kaggle.
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def face_stats(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return 0, 0.0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40)
    )
    num = len(faces)
    if num == 0:
        return 0, 0.0
    # area of largest face box relative to image area (helps robustness)
    h, w = gray.shape
    areas = [(fw * fh) / float(w * h) for (x, y, fw, fh) in faces]
    return num, float(max(areas))

num_faces_list = []
max_face_area_list = []

for p in tqdm(df["image_path"].fillna("").tolist()):
    if not p:
        num_faces_list.append(0)
        max_face_area_list.append(0.0)
        continue
    n, a = face_stats(p)
    num_faces_list.append(n)
    max_face_area_list.append(a)

df["num_faces"] = num_faces_list
df["max_face_area"] = max_face_area_list

100%|██████████| 1303/1303 [04:07<00:00,  5.26it/s]


In [13]:
out_path = "/kaggle/working/train_with_faces.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

# Show useful summaries
print("\nFace count summary by language:")
print(df.groupby("lang")["num_faces"].describe())

print("\nMalayalam Intersection vs num_faces:")
mal = df[df["lang"]=="malayalam"]
print(pd.crosstab(mal["target_internal"], mal["num_faces"]>=2, rownames=["target_internal"], colnames=[">=2 faces"]))

Saved: /kaggle/working/train_with_faces.csv

Face count summary by language:
           count      mean       std  min  25%  50%  75%   max
lang                                                          
malayalam  500.0  2.598000  2.307172  0.0  1.0  2.0  3.0  14.0
tamil      803.0  2.283935  2.131951  0.0  1.0  2.0  3.0  20.0

Malayalam Intersection vs num_faces:
>=2 faces        False  True 
target_internal              
INTERSECTION        12     41
PARTY               40     80
PERSON             137    190


In [14]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/kaggle/working/train_with_faces.csv")

# simple feature set for now
X = df[["num_faces", "max_face_area"]].copy()

def eval_task(y, name):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000))
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X, y, cv=cv, scoring="f1_macro")
    print(f"{name} macro-F1: {scores.mean():.4f} ± {scores.std():.4f}")

eval_task(df["stance_internal"], "STANCE (SUPPORT vs TROLL)")
eval_task(df["target_internal"], "TARGET (PERSON/PARTY/INTERSECTION)")

STANCE (SUPPORT vs TROLL) macro-F1: 0.4727 ± 0.0001
TARGET (PERSON/PARTY/INTERSECTION) macro-F1: 0.2826 ± 0.0005


In [ ]:
# -----------------------------
# 0) Switches (turn on/off parts)
# -----------------------------
RUN_BASELINE_FACE_ONLY   = True
RUN_QWEN_VL              = True      # heavy GPU
RUN_CLIP_EMBEDS          = True      # medium-heavy GPU
RUN_EXPORT_QWEN_TEST_CSV = True
RUN_EXPORT_CLIP_TEST_CSV = True


In [ ]:
# -----------------------------
# 2) Imports + utilities
# -----------------------------
import os, re, json
import numpy as np
import pandas as pd
from tqdm import tqdm

RESULTS = {}  # collect all scores + notes in one place


def add_result(key, value):
    RESULTS[key] = value
    print(f"[RESULT] {key}: {value}")


In [15]:
# -----------------------------
# 3) Paths (train/test)
# -----------------------------
TRAIN_WITH_FACES_CSV = "/kaggle/working/train_with_faces.csv"

TAMIL_TEST_IMG_DIR = "/kaggle/input/shared-task-meme-test/test/Tamil_test/Tamil_Test_images"
MAL_TEST_IMG_DIR   = "/kaggle/input/shared-task-meme-test/test/Malaylam_test/Malaylam_Test_images"

TAMIL_TEST_CSV = "/kaggle/input/shared-task-meme-test/test/Tamil_test/Tamil_test_labels.csv"
MAL_TEST_CSV   = "/kaggle/input/shared-task-meme-test/test/Malaylam_test/Malayalam_Test_labels.csv"



In [16]:
# -----------------------------
# 4) Load train table
# -----------------------------
df = pd.read_csv(TRAIN_WITH_FACES_CSV)
assert "image_path" in df.columns, "train_with_faces.csv must contain image_path"
assert "stance_internal" in df.columns and "target_internal" in df.columns, "need stance_internal, target_internal"
assert "num_faces" in df.columns and "max_face_area" in df.columns, "need face columns num_faces, max_face_area"

print("Train rows:", len(df))
print("Columns:", list(df.columns)[:20], "...")


Train rows: 1303
Columns: ['lang', 'image_path', 'Level 1', 'Level 2', 'stance_internal', 'target_internal', 'num_faces', 'max_face_area'] ...


In [17]:
# ============================================================
# A) Approach 1: Face-only baseline (LogReg + StandardScaler)
# ============================================================
if RUN_BASELINE_FACE_ONLY:
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression

    X_face = df[["num_faces", "max_face_area"]].copy()

    def eval_face_only(y, name):
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(max_iter=2000))
        ])
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(clf, X_face, y, cv=cv, scoring="f1_macro")
        mean, std = float(scores.mean()), float(scores.std())
        print(f"{name} macro-F1: {mean:.4f} ± {std:.4f}")
        return mean, std

    m, s = eval_face_only(df["stance_internal"], "FACE-ONLY STANCE (SUPPORT vs TROLL)")
    add_result("face_only_stance_macro_f1_mean_std", (m, s))

    m, s = eval_face_only(df["target_internal"], "FACE-ONLY TARGET (PERSON/PARTY/INTERSECTION)")
    add_result("face_only_target_macro_f1_mean_std", (m, s))



FACE-ONLY STANCE (SUPPORT vs TROLL) macro-F1: 0.4727 ± 0.0001
[RESULT] face_only_stance_macro_f1_mean_std: (0.4726830167803799, 5.420224398560534e-05)
FACE-ONLY TARGET (PERSON/PARTY/INTERSECTION) macro-F1: 0.2826 ± 0.0005
[RESULT] face_only_target_macro_f1_mean_std: (0.2826408017347477, 0.0005425229661060011)


In [21]:
# ============================================================
# B) Make a clean val_df split (needed for Qwen evaluation)
# ============================================================
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["stance_internal"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("VAL stance counts:\n", val_df["stance_internal"].value_counts())
print("VAL target counts:\n", val_df["target_internal"].value_counts())
print("VAL joint counts:\n", (val_df["stance_internal"]+"|"+val_df["target_internal"]).value_counts())




VAL stance counts:
 stance_internal
TROLL      234
SUPPORT     27
Name: count, dtype: int64
VAL target counts:
 target_internal
PERSON          202
PARTY            47
INTERSECTION     12
Name: count, dtype: int64
VAL joint counts:
 TROLL|PERSON          181
TROLL|PARTY            41
SUPPORT|PERSON         21
TROLL|INTERSECTION     12
SUPPORT|PARTY           6
Name: count, dtype: int64


In [22]:
# ============================================================
# C) Approach 2: Qwen2.5-VL prompt inference (w/ and w/o hints)
# ============================================================
if RUN_QWEN_VL:
    import torch
    from PIL import Image, ImageFile
    ImageFile.LOAD_TRUNCATED_IMAGES = True

    from transformers import AutoProcessor, AutoModelForImageTextToText

    MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # consider 3B if you OOM
    VISION_PREFIX = "<|vision_start|><|image_pad|><|vision_end|>\n"

    PROMPT = """Classify the political meme into two levels.

Return ONLY this JSON:
{"stance":"SUPPORT|TROLL","target":"PERSON|PARTY|INTERSECTION"}

Rules:
- SUPPORT: praise/endorsement
- TROLL: oppose/mock/criticize
- PERSON: about an individual
- PARTY: about a party/symbol/flag
- INTERSECTION: multiple targets (multi-person or person+party)
"""

    def parse_json(text):
        try:
            return json.loads(text)
        except Exception:
            m = re.search(r"\{.*\}", text, flags=re.S)
            if m:
                try:
                    return json.loads(m.group(0))
                except Exception:
                    return None
        return None

    def normalize(pred):
        stance = str(pred.get("stance", "")).strip().upper()
        target = str(pred.get("target", "")).strip().upper()
        if stance not in ["SUPPORT", "TROLL"]:
            stance = "TROLL"
        if target not in ["PERSON", "PARTY", "INTERSECTION"]:
            target = "PERSON"
        return stance, target

    print("Loading Qwen processor/model...")
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print("Qwen loaded.")

    @torch.inference_mode()
    def qwen_predict(image_path):
        text = VISION_PREFIX + PROMPT
        try:
            image = Image.open(image_path).convert("RGB")
            if max(image.size) > 1280:
                image.thumbnail((1280, 1280))

            inputs = processor(
                text=[text],
                images=[image],
                return_tensors="pt",
                padding=True
            ).to(model.device)

            out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
            decoded = processor.batch_decode(out, skip_special_tokens=True)[0].strip()

            pred = parse_json(decoded) or {}
            stance, target = normalize(pred)
            return stance, target, decoded
        except Exception as e:
            return "TROLL", "PERSON", f"__ERROR__ {type(e).__name__}: {e}"

    @torch.inference_mode()
    def qwen_predict_with_hints(image_path, num_faces=None, max_face_area=None):
        image = Image.open(image_path).convert("RGB")
        if max(image.size) > 1280:
            image.thumbnail((1280, 1280))

        hints = ""
        if num_faces is not None:
            hints += f"\nDetected hints:\n- faces_detected: {int(num_faces)}"
        if max_face_area is not None:
            hints += f"\n- largest_face_area_ratio: {float(max_face_area):.4f}"

        text = VISION_PREFIX + (PROMPT + hints)

        inputs = processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding=True
        ).to(model.device)

        out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
        decoded = processor.batch_decode(out, skip_special_tokens=True)[0].strip()

        pred = parse_json(decoded) or {}
        stance, target = normalize(pred)
        return stance, target, decoded

    # ---- Evaluate on val_df (no hints) ----
    from sklearn.metrics import f1_score

    pred_stance_vl, pred_target_vl = [], []
    bad = 0

    for p in tqdm(val_df["image_path"].tolist(), total=len(val_df), desc="QWEN val (no hints)"):
        stance, target, raw = qwen_predict(p)
        if isinstance(raw, str) and raw.startswith("__ERROR__"):
            bad += 1
        pred_stance_vl.append(stance)
        pred_target_vl.append(target)

    val_eval = val_df.copy()
    val_eval["pred_stance_vl"] = pred_stance_vl
    val_eval["pred_target_vl"] = pred_target_vl

    stance_f1 = float(f1_score(val_eval["stance_internal"], val_eval["pred_stance_vl"], average="macro"))
    target_f1 = float(f1_score(val_eval["target_internal"], val_eval["pred_target_vl"], average="macro"))

    add_result("qwen_val_no_hints_stance_macro_f1", stance_f1)
    add_result("qwen_val_no_hints_target_macro_f1", target_f1)
    add_result("qwen_val_no_hints_bad_images", int(bad))

    # ---- Evaluate on val_df (with hints) ----
    pred_stance_h, pred_target_h = [], []
    bad_h = 0

    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="QWEN val (with hints)"):
        stance, target, raw = qwen_predict_with_hints(
            row["image_path"],
            num_faces=row["num_faces"],
            max_face_area=row["max_face_area"]
        )
        if isinstance(raw, str) and raw.startswith("__ERROR__"):
            bad_h += 1
        pred_stance_h.append(stance)
        pred_target_h.append(target)

    val_eval["pred_stance_vl_h"] = pred_stance_h
    val_eval["pred_target_vl_h"] = pred_target_h

    stance_f1_h = float(f1_score(val_eval["stance_internal"], val_eval["pred_stance_vl_h"], average="macro"))
    target_f1_h = float(f1_score(val_eval["target_internal"], val_eval["pred_target_vl_h"], average="macro"))

    add_result("qwen_val_with_hints_stance_macro_f1", stance_f1_h)
    add_result("qwen_val_with_hints_target_macro_f1", target_f1_h)
    add_result("qwen_val_with_hints_bad_images", int(bad_h))



Loading Qwen processor/model...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Qwen loaded.


QWEN val (no hints): 100%|██████████| 261/261 [12:28<00:00,  2.87s/it]


[RESULT] qwen_val_no_hints_stance_macro_f1: 0.4727272727272727
[RESULT] qwen_val_no_hints_target_macro_f1: 0.2908567314614831
[RESULT] qwen_val_no_hints_bad_images: 0


QWEN val (with hints): 100%|██████████| 261/261 [13:04<00:00,  3.01s/it]

[RESULT] qwen_val_with_hints_stance_macro_f1: 0.4727272727272727
[RESULT] qwen_val_with_hints_target_macro_f1: 0.2908567314614831
[RESULT] qwen_val_with_hints_bad_images: 0


In [23]:
# ============================================================
# D) Approach 3: CLIP embeddings + face feats + LogisticRegression CV
# ============================================================
X_clip_face = None  # will be built here and reused later

if RUN_CLIP_EMBEDS:
    import torch
    from PIL import Image
    import open_clip
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_clip, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32",
        pretrained="laion2b_s34b_b79k"
    )
    model_clip = model_clip.to(device).eval()

    @torch.inference_mode()
    def clip_embed(path):
        img = Image.open(path).convert("RGB")
        img = preprocess(img).unsqueeze(0).to(device)
        feat = model_clip.encode_image(img)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        return feat.squeeze(0).detach().cpu().numpy()

    # Compute embeddings for ALL train images
    embs = []
    for p in tqdm(df["image_path"].tolist(), desc="CLIP embeds train"):
        embs.append(clip_embed(p))
    embs = np.vstack(embs)

    face_feats = df[["num_faces", "max_face_area"]].to_numpy().astype(np.float32)
    X_clip_face = np.hstack([embs, face_feats])

    # ---- stance CV ----
    y_stance = df["stance_internal"].values
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf_stance = LogisticRegression(max_iter=2000)
    pred_stance = cross_val_predict(clf_stance, X_clip_face, y_stance, cv=skf)
    stance_f1 = float(f1_score(y_stance, pred_stance, average="macro"))
    add_result("clip_face_logreg_cv_stance_macro_f1", stance_f1)

    # ---- target CV ----
    y_target = df["target_internal"].values
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf_target = LogisticRegression(max_iter=2000)
    pred_target = cross_val_predict(clf_target, X_clip_face, y_target, cv=skf2)
    target_f1 = float(f1_score(y_target, pred_target, average="macro"))
    add_result("clip_face_logreg_cv_target_macro_f1", target_f1)



open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP embeds train: 100%|██████████| 1303/1303 [00:37<00:00, 34.78it/s]


[RESULT] clip_face_logreg_cv_stance_macro_f1: 0.5428833945873706
[RESULT] clip_face_logreg_cv_target_macro_f1: 0.31277975287395365


In [24]:
# ============================================================
# E) Approach 4: CLIP+face + LinearSVC per language (CV)
# ============================================================
if RUN_CLIP_EMBEDS:
    from sklearn.svm import LinearSVC
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.metrics import f1_score

    def cv_score(X, y, model):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        pred = cross_val_predict(model, X, y, cv=skf)
        return float(f1_score(y, pred, average="macro"))

    for lang in ["tamil", "malayalam"]:
        mask = (df["lang"].values == lang)
        Xsub = X_clip_face[mask]
        y_st = df.loc[mask, "stance_internal"].values
        y_tg = df.loc[mask, "target_internal"].values

        stance_score = cv_score(Xsub, y_st, LinearSVC(class_weight="balanced"))
        target_score = cv_score(Xsub, y_tg, LinearSVC(class_weight="balanced"))

        add_result(f"clip_face_linearsvc_cv_{lang}_stance_macro_f1", stance_score)
        add_result(f"clip_face_linearsvc_cv_{lang}_target_macro_f1", target_score)



[RESULT] clip_face_linearsvc_cv_tamil_stance_macro_f1: 0.7987705648095311
[RESULT] clip_face_linearsvc_cv_tamil_target_macro_f1: 0.5776859504132231


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

[RESULT] clip_face_linearsvc_cv_malayalam_stance_macro_f1: 0.6513319120957941
[RESULT] clip_face_linearsvc_cv_malayalam_target_macro_f1: 0.48225217301454154


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [25]:
# ============================================================
# F) Mapping functions for test CSVs (Tamil / Malayalam)
# ============================================================
def map_to_tamil(stance, target):
    # Tamil allowed Level1: Support/Praise, Troll/Oppose
    # Tamil Level2: Support for party/person, Troll/Oppose Against Party/Person
    if stance == "SUPPORT":
        lvl1 = "Support/Praise"
        lvl2 = "Support for person" if target != "PARTY" else "Support for party"
    else:
        lvl1 = "Troll/Oppose"
        lvl2 = "Troll/Oppose Against Person" if target != "PARTY" else "Troll/Oppose Against Party"
    return lvl1, lvl2

def map_to_malayalam(stance, target):
    # Malayalam allowed Level1: SUPPORT, TROLL/ OPPOSE
    # Malayalam Level2: Against individual person, Against party, Intersection,
    #                  Support for individual person, Support for party
    if stance == "SUPPORT":
        lvl1 = "SUPPORT"
        if target == "PERSON":
            lvl2 = "Support for individual person"
        elif target == "PARTY":
            lvl2 = "Support for party"
        else:
            lvl2 = "Intersection"
    else:
        lvl1 = "TROLL/ OPPOSE"
        if target == "PERSON":
            lvl2 = "Against individual person"
        elif target == "PARTY":
            lvl2 = "Against party"
        else:
            lvl2 = "Intersection"
    return lvl1, lvl2




In [26]:
# ============================================================
# G) Export test predictions (Qwen prompt approach)
# ============================================================
if RUN_QWEN_VL and RUN_EXPORT_QWEN_TEST_CSV:
    import pandas as pd

    # ---- Tamil ----
    tam = pd.read_csv(TAMIL_TEST_CSV)
    pred_lvl1, pred_lvl2, raw_out = [], [], []
    bad = 0

    for _, r in tqdm(tam.iterrows(), total=len(tam), desc="QWEN Tamil test"):
        img_name = str(r["Image_name"]).strip()
        img_path = os.path.join(TAMIL_TEST_IMG_DIR, img_name)

        stance, target, raw = qwen_predict(img_path)

        if isinstance(raw, str) and raw.startswith("__ERROR__"):
            bad += 1
            stance, target = "TROLL", "PERSON"

        lvl1, lvl2 = map_to_tamil(stance, target)
        pred_lvl1.append(lvl1)
        pred_lvl2.append(lvl2)
        raw_out.append(raw)

    tam["Level1"] = pred_lvl1
    tam["Level2"] = pred_lvl2
    tam["raw_model_output"] = raw_out

    out_tam = "/kaggle/working/Tamil_test_predictions_QWEN.csv"
    tam.to_csv(out_tam, index=False)
    add_result("qwen_tamil_test_csv", out_tam)
    add_result("qwen_tamil_test_bad_images", int(bad))

    # ---- Malayalam ----
    mal = pd.read_csv(MAL_TEST_CSV)
    pred_l1, pred_l2, bad2 = [], [], 0

    for _, r in tqdm(mal.iterrows(), total=len(mal), desc="QWEN Malayalam test"):
        meme_id = int(r["meme_id"])
        img_path = os.path.join(MAL_TEST_IMG_DIR, f"{meme_id}.jpg")

        stance, target, raw = qwen_predict(img_path)
        if isinstance(raw, str) and raw.startswith("__ERROR__"):
            bad2 += 1
            stance, target = "TROLL", "PERSON"

        lvl1, lvl2 = map_to_malayalam(stance, target)
        pred_l1.append(lvl1)
        pred_l2.append(lvl2)

    mal["Level 1"] = pred_l1
    mal["Level 2"] = pred_l2

    out_mal = "/kaggle/working/Malayalam_test_predictions_QWEN.csv"
    mal.to_csv(out_mal, index=False)
    add_result("qwen_malayalam_test_csv", out_mal)
    add_result("qwen_malayalam_test_bad_images", int(bad2))




QWEN Tamil test: 100%|██████████| 201/201 [07:34<00:00,  2.26s/it]


[RESULT] qwen_tamil_test_csv: /kaggle/working/Tamil_test_predictions_QWEN.csv
[RESULT] qwen_tamil_test_bad_images: 0


QWEN Malayalam test: 100%|██████████| 100/100 [06:01<00:00,  3.61s/it]

[RESULT] qwen_malayalam_test_csv: /kaggle/working/Malayalam_test_predictions_QWEN.csv
[RESULT] qwen_malayalam_test_bad_images: 0


In [27]:
# ============================================================
# H) Export test predictions (CLIP+LinearSVC per language)
# ============================================================
if RUN_CLIP_EMBEDS and RUN_EXPORT_CLIP_TEST_CSV:
    import cv2
    from PIL import Image
    import torch
    from sklearn.svm import LinearSVC

    # Face detector (fast Haar cascade)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    def face_stats_cv2(path):
        img = cv2.imread(path)
        if img is None:
            return 0, 0.0
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
        num = len(faces)
        if num == 0:
            return 0, 0.0
        h, w = gray.shape
        areas = [(fw * fh) / (w * h) for (x, y, fw, fh) in faces]
        return num, float(max(areas))

    # Reuse model_clip, preprocess, device, clip_embed from above
    # Train 4 models (2 tasks x 2 languages) on FULL train
    def train_models_for_lang(lang):
        mask = (df["lang"].values == lang)
        Xsub = X_clip_face[mask]
        y_stance = df.loc[mask, "stance_internal"].values
        y_target = df.loc[mask, "target_internal"].values

        m_stance = LinearSVC(class_weight="balanced")
        m_target = LinearSVC(class_weight="balanced")

        m_stance.fit(Xsub, y_stance)
        m_target.fit(Xsub, y_target)
        return m_stance, m_target

    m_ta_stance, m_ta_target = train_models_for_lang("tamil")
    m_ml_stance, m_ml_target = train_models_for_lang("malayalam")
    print("Trained 4 CLIP+SVC models.")

    def build_test_X(image_paths):
        embs_test = []
        face_test = []
        for p in tqdm(image_paths, desc="Build test X"):
            embs_test.append(clip_embed(p))
            n, a = face_stats_cv2(p)
            face_test.append([n, a])
        embs_test = np.vstack(embs_test)
        face_test = np.array(face_test, dtype=np.float32)
        return np.hstack([embs_test, face_test])

    # ---- Tamil ----
    tam = pd.read_csv(TAMIL_TEST_CSV)
    tam_paths = [os.path.join(TAMIL_TEST_IMG_DIR, str(n).strip()) for n in tam["Image_name"]]
    X_tam_test = build_test_X(tam_paths)

    pred_stance = m_ta_stance.predict(X_tam_test)
    pred_target = m_ta_target.predict(X_tam_test)

    lvl1, lvl2 = [], []
    for s, t in zip(pred_stance, pred_target):
        a, b = map_to_tamil(s, t)
        lvl1.append(a); lvl2.append(b)

    tam["Level1"] = lvl1
    tam["Level2"] = lvl2

    out_tam = "/kaggle/working/Tamil_test_predictions_CLIP_SVC.csv"
    tam.to_csv(out_tam, index=False)
    add_result("clip_svc_tamil_test_csv", out_tam)

    # ---- Malayalam ----
    mal = pd.read_csv(MAL_TEST_CSV)
    mal_paths = [os.path.join(MAL_TEST_IMG_DIR, f"{int(i)}.jpg") for i in mal["meme_id"]]
    X_mal_test = build_test_X(mal_paths)

    pred_stance = m_ml_stance.predict(X_mal_test)
    pred_target = m_ml_target.predict(X_mal_test)

    lvl1, lvl2 = [], []
    for s, t in zip(pred_stance, pred_target):
        a, b = map_to_malayalam(s, t)
        lvl1.append(a); lvl2.append(b)

    mal["Level 1"] = lvl1
    mal["Level 2"] = lvl2

    out_mal = "/kaggle/working/Malayalam_test_predictions_CLIP_SVC.csv"
    mal.to_csv(out_mal, index=False)
    add_result("clip_svc_malayalam_test_csv", out_mal)



/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Trained 4 CLIP+SVC models.


Build test X: 100%|██████████| 201/201 [00:34<00:00,  5.75it/s]


[RESULT] clip_svc_tamil_test_csv: /kaggle/working/Tamil_test_predictions_CLIP_SVC.csv


Build test X: 100%|██████████| 100/100 [00:28<00:00,  3.54it/s]

[RESULT] clip_svc_malayalam_test_csv: /kaggle/working/Malayalam_test_predictions_CLIP_SVC.csv


In [28]:
# ============================================================
# I) Final: print ALL results collected
# ============================================================
print("\n" + "="*60)
print("ALL RESULTS (collected):")
print("="*60)

# Pretty print
for k in sorted(RESULTS.keys()):
    print(f"{k}: {RESULTS[k]}")

# Optional: show as a small dataframe
try:
    res_df = pd.DataFrame([{"metric": k, "value": RESULTS[k]} for k in sorted(RESULTS.keys())])
    display(res_df)
except Exception:
    pass



ALL RESULTS (collected):
clip_face_linearsvc_cv_malayalam_stance_macro_f1: 0.6513319120957941
clip_face_linearsvc_cv_malayalam_target_macro_f1: 0.48225217301454154
clip_face_linearsvc_cv_tamil_stance_macro_f1: 0.7987705648095311
clip_face_linearsvc_cv_tamil_target_macro_f1: 0.5776859504132231
clip_face_logreg_cv_stance_macro_f1: 0.5428833945873706
clip_face_logreg_cv_target_macro_f1: 0.31277975287395365
clip_svc_malayalam_test_csv: /kaggle/working/Malayalam_test_predictions_CLIP_SVC.csv
clip_svc_tamil_test_csv: /kaggle/working/Tamil_test_predictions_CLIP_SVC.csv
face_only_stance_macro_f1_mean_std: (0.4726830167803799, 5.420224398560534e-05)
face_only_target_macro_f1_mean_std: (0.2826408017347477, 0.0005425229661060011)
qwen_malayalam_test_bad_images: 0
qwen_malayalam_test_csv: /kaggle/working/Malayalam_test_predictions_QWEN.csv
qwen_tamil_test_bad_images: 0
qwen_tamil_test_csv: /kaggle/working/Tamil_test_predictions_QWEN.csv
qwen_val_no_hints_bad_images: 0
qwen_val_no_hints_stance_mac

,metric,value
0,clip_face_linearsvc_cv_malayalam_stance_macro_f1,0.651332
1,clip_face_linearsvc_cv_malayalam_target_macro_f1,0.482252
2,clip_face_linearsvc_cv_tamil_stance_macro_f1,0.798771
3,clip_face_linearsvc_cv_tamil_target_macro_f1,0.577686
4,clip_face_logreg_cv_stance_macro_f1,0.542883
5,clip_face_logreg_cv_target_macro_f1,0.31278
6,clip_svc_malayalam_test_csv,/kaggle/working/Malayalam_test_predictions_CLI...
7,clip_svc_tamil_test_csv,/kaggle/working/Tamil_test_predictions_CLIP_SV...
8,face_only_stance_macro_f1_mean_std,"(0.4726830167803799, 5.420224398560534e-05)"
9,face_only_target_macro_f1_mean_std,"(0.2826408017347477, 0.0005425229661060011)"


In [11]:
# ============================================================================
# CELL 9: Evaluate combined features with cross-validation
# ============================================================================
print("\n>>> PHASE 6: Evaluating CLIP + face features...")

y_stance = df["stance_internal"].values
y_target = df["target_internal"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_stance = LogisticRegression(max_iter=2000)
pred_stance_cv = cross_val_predict(clf_stance, X, y_stance, cv=skf)

print(f"STANCE macro-F1: {f1_score(y_stance, pred_stance_cv, average='macro'):.4f}")

clf_target = LogisticRegression(max_iter=2000)
pred_target_cv = cross_val_predict(clf_target, X, y_target, cv=skf)

print(f"TARGET macro-F1: {f1_score(y_target, pred_target_cv, average='macro'):.4f}")



>>> PHASE 6: Evaluating CLIP + face features...
STANCE macro-F1: 0.5429
TARGET macro-F1: 0.3128


In [12]:
# ============================================================================
# CELL 10: Train final models (per language, LinearSVC)
# ============================================================================
print("\n>>> PHASE 7: Training final models (per language)...")

def train_models_for_lang(lang):
    mask = (df["lang"].values == lang)
    Xsub = X[mask]
    y_stance = df.loc[mask, "stance_internal"].values
    y_target = df.loc[mask, "target_internal"].values

    m_stance = LinearSVC(class_weight="balanced", max_iter=5000)
    m_target = LinearSVC(class_weight="balanced", max_iter=5000)

    m_stance.fit(Xsub, y_stance)
    m_target.fit(Xsub, y_target)
    
    print(f"  {lang.upper()} models trained")
    return m_stance, m_target

m_ta_stance, m_ta_target = train_models_for_lang("tamil")
m_ml_stance, m_ml_target = train_models_for_lang("malayalam")

print("✅ All models trained")




>>> PHASE 7: Training final models (per language)...
  TAMIL models trained


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  MALAYALAM models trained
✅ All models trained


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [13]:
# ============================================================================
# CELL 11: Define mapping functions for output
# ============================================================================
print("\n>>> PHASE 8: Defining output mappings...")

def map_to_tamil(stance, target):
    if stance == "SUPPORT":
        lvl1 = "Support/Praise"
        if target == "PARTY":
            lvl2 = "Support for party"
        else:
            lvl2 = "Support for person"
    else:
        lvl1 = "Troll/Oppose"
        if target == "PARTY":
            lvl2 = "Troll/Oppose Against Party"
        else:
            lvl2 = "Troll/Oppose Against Person"
    return lvl1, lvl2

def map_to_malayalam(stance, target):
    if stance == "SUPPORT":
        lvl1 = "SUPPORT"
        if target == "PERSON":
            lvl2 = "Support for individual person"
        elif target == "PARTY":
            lvl2 = "Support for party"
        else:
            lvl2 = "Intersection"
    else:
        lvl1 = "TROLL/ OPPOSE"
        if target == "PERSON":
            lvl2 = "Against individual person"
        elif target == "PARTY":
            lvl2 = "Against party"
        else:
            lvl2 = "Intersection"
    return lvl1, lvl2

print("✅ Mapping functions defined")




>>> PHASE 8: Defining output mappings...
✅ Mapping functions defined


In [14]:
# ============================================================================
# CELL 12: Face detection and CLIP embedding for test data
# ============================================================================
print("\n>>> PHASE 9: Processing test data...")

def face_stats_cv2(path):
    img = cv2.imread(path)
    if img is None:
        return 0, 0.0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    num = len(faces)
    if num == 0:
        return 0, 0.0
    h, w = gray.shape
    areas = [(fw * fh) / (w * h) for (x, y, fw, fh) in faces]
    return num, float(max(areas))

def build_test_X(image_paths):
    """Build feature matrix for test images"""
    embs_test = []
    face_test = []
    
    for p in tqdm(image_paths, desc="Building test features"):
        try:
            embs_test.append(clip_embed(p))
            n, a = face_stats_cv2(p)
            face_test.append([n, a])
        except Exception as e:
            print(f"Warning: Error processing {p}: {e}")
            # Default values on error
            embs_test.append(np.zeros(512))  # CLIP ViT-B-32 is 512-dim
            face_test.append([0, 0.0])
    
    embs_test = np.vstack(embs_test)
    face_test = np.array(face_test, dtype=np.float32)
    return np.hstack([embs_test, face_test])




>>> PHASE 9: Processing test data...


In [15]:
# ============================================================================
# CELL 13: Predict on Tamil test set
# ============================================================================
print("\n>>> PHASE 10: Tamil test predictions...")

tam = pd.read_csv(TAMIL_TEST_CSV)
tam_paths = [os.path.join(TAMIL_TEST_IMG_DIR, str(n).strip()) for n in tam["Image_name"]]

X_tam_test = build_test_X(tam_paths)

pred_stance_tam = m_ta_stance.predict(X_tam_test)
pred_target_tam = m_ta_target.predict(X_tam_test)

lvl1_tam, lvl2_tam = [], []
for s, t in zip(pred_stance_tam, pred_target_tam):
    a, b = map_to_tamil(s, t)
    lvl1_tam.append(a)
    lvl2_tam.append(b)

tam["Level1"] = lvl1_tam
tam["Level2"] = lvl2_tam

out_tam_pred = "/kaggle/working/Tamil_test_predictions.csv"
tam.to_csv(out_tam_pred, index=False)
print(f"✅ Saved: {out_tam_pred}")




>>> PHASE 10: Tamil test predictions...


Building test features: 100%|██████████| 201/201 [00:33<00:00,  6.04it/s]

✅ Saved: /kaggle/working/Tamil_test_predictions.csv


In [16]:
# ============================================================================
# CELL 14: Predict on Malayalam test set
# ============================================================================
print("\n>>> PHASE 11: Malayalam test predictions...")

mal = pd.read_csv(MAL_TEST_CSV)
mal_paths = [os.path.join(MAL_TEST_IMG_DIR, f"{int(i)}.jpg") for i in mal["meme_id"]]

X_mal_test = build_test_X(mal_paths)

pred_stance_mal = m_ml_stance.predict(X_mal_test)
pred_target_mal = m_ml_target.predict(X_mal_test)

lvl1_mal, lvl2_mal = [], []
for s, t in zip(pred_stance_mal, pred_target_mal):
    a, b = map_to_malayalam(s, t)
    lvl1_mal.append(a)
    lvl2_mal.append(b)

mal["Level 1"] = lvl1_mal
mal["Level 2"] = lvl2_mal

out_mal_pred = "/kaggle/working/Malayalam_test_predictions.csv"
mal.to_csv(out_mal_pred, index=False)
print(f"✅ Saved: {out_mal_pred}")



>>> PHASE 11: Malayalam test predictions...


Building test features: 100%|██████████| 100/100 [00:28<00:00,  3.53it/s]

✅ Saved: /kaggle/working/Malayalam_test_predictions.csv


In [17]:
# ============================================================================
# CELL 15: Export final submission files
# ============================================================================
print("\n>>> PHASE 12: Exporting submission files...")

tam_submit = "/kaggle/working/Tamil_submit.csv"
tam[["Image_id", "Image_name", "Level1", "Level2"]].to_csv(tam_submit, index=False)
print(f"✅ Saved: {tam_submit}")

mal_submit = "/kaggle/working/Malayalam_submit.csv"
mal[["meme_id", "Level 1", "Level 2"]].to_csv(mal_submit, index=False)
print(f"✅ Saved: {mal_submit}")




>>> PHASE 12: Exporting submission files...
✅ Saved: /kaggle/working/Tamil_submit.csv
✅ Saved: /kaggle/working/Malayalam_submit.csv


In [18]:
# ============================================================================
# CELL 16: Validation
# ============================================================================
print("\n>>> PHASE 13: Validating outputs...")

print("\nTamil output labels:")
print(f"  Level1 values: {set(tam['Level1'])}")
print(f"  Level2 values: {set(tam['Level2'])}")

print("\nMalayalam output labels:")
print(f"  Level 1 values: {set(mal['Level 1'])}")
print(f"  Level 2 values: {set(mal['Level 2'])}")

print("\n" + "="*70)
print("✅ PIPELINE COMPLETE!")
print("="*70)
print(f"\nFinal outputs saved:")
print(f"  1. {tam_submit}")
print(f"  2. {mal_submit}")
print(f"\nReady for Kaggle submission!")



>>> PHASE 13: Validating outputs...

Tamil output labels:
  Level1 values: {'Support/Praise', 'Troll/Oppose'}
  Level2 values: {'Support for person', 'Troll/Oppose Against Party', 'Support for party', 'Troll/Oppose Against Person'}

Malayalam output labels:
  Level 1 values: {'TROLL/ OPPOSE', 'SUPPORT'}
  Level 2 values: {'Intersection', 'Against party', 'Support for party', 'Against individual person', 'Support for individual person'}

✅ PIPELINE COMPLETE!

Final outputs saved:
  1. /kaggle/working/Tamil_submit.csv
  2. /kaggle/working/Malayalam_submit.csv

Ready for Kaggle submission!


In [29]:
tamil_clip = pd.read_csv("/kaggle/working/Tamil_test_predictions_CLIP_SVC.csv")
tamil_clip

,Image_id,Image_name,Level1,Level2
0,1,001.jpg,Troll/Oppose,Troll/Oppose Against Party
1,6,006.jpg,Troll/Oppose,Troll/Oppose Against Person
2,15,015.jpg,Troll/Oppose,Troll/Oppose Against Party
3,23,023.jpg,Troll/Oppose,Troll/Oppose Against Person
4,24,024.jpg,Troll/Oppose,Troll/Oppose Against Person
...,...,...,...,...
196,974,974.jpg,Support/Praise,Support for party
197,979,979.jpg,Troll/Oppose,Troll/Oppose Against Party
198,990,990.jpg,Troll/Oppose,Troll/Oppose Against Person
199,993,993.jpg,Troll/Oppose,Troll/Oppose Against Person


In [ ]:
bjbjbjm